# Task 1 — Weather variables worth investigating

本 notebook 研究五個 forecast families 的觀測資訊關係，提出後續研究問題。
保留既有 temperature、RH、rainfall 選題；不預設前三名，不輸出綜合重要性排名。

**尚無真實資料模型結果。請依序 Run All，再回傳最後產生的 results ZIP。**
所有 regression fitting、selection、temporal validation、plots 均在本 notebook。
CPU 即可執行；不使用 CUDA、PyTorch 或 GPU。

這是 **ex-post cross-variable reconstruction**，不是 forecast issuance-time prediction。
同日 observations 是此任務合法的 inputs，不能轉用作 operational forecast features。
2025 已用於先前 EDA，只是 chronological evaluation，並非全研究 untouched test。

## 1. Setup

從 workspace 執行 README 的安裝步驟，選擇對應 Jupyter kernel。
Notebook 預設讀取已凍結、hash-verified inputs，不需要 database bridge。
正常研究請保留 SYNTHETIC=False；True 只用於軟件驗證。
每次執行建立獨立 results directory，不覆寫 inputs 或既有 results。

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "requirements-variable-selection.txt").exists())
DATA_DIR = ROOT / "data/derived/task1_variable_selection_20261117"
INPUT_DIR = DATA_DIR / "inputs"
SYNTHETIC = False
SEED = 20261117
N_BOOTSTRAP = 1000
N_ALPHA = 40
TRACE_VALUES = {"primary": 0.025, "trace_lower": 0.0, "trace_upper": 0.05}
MIN_TRAIN = 180
MIN_TEST = 60
GAP_DAYS = 7

In [ ]:
import hashlib
import importlib.metadata
import itertools
import json
import math
import shutil
import warnings
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display
from scipy.special import logsumexp
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

FAMILIES = {
    "temperature": ["tmax", "tmin"],
    "humidity": ["rh_min", "rh_max"],
    "rainfall": ["rainfall"],
    "wind": ["wind_speed"],
    "weather": ["cloud", "sunshine"],
}
CHANNELS = [c for group in FAMILIES.values() for c in group]
FAMILY_OF = {c: f for f, group in FAMILIES.items() for c in group}
UNITS = dict(tmax="°C", tmin="°C", rh_min="%", rh_max="%",
             rainfall="mm", wind_speed="km/h", cloud="%", sunshine="hours")
STATIONS = {c: "Hong Kong Observatory" for c in CHANNELS}
STATIONS.update(wind_speed="Hong Kong International Airport", sunshine="King's Park")
CALENDAR = pd.date_range("2022-01-01", "2025-12-31", name="date_hkt")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.bbox": "tight"})
COLORS = {"f_selected_ols": "#356a9a", "lasso": "#ad5729", "bayesian_bma": "#397d62"}


def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def json_save(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                          encoding="utf-8")


def load_frozen(folder):
    manifest_path = folder / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError("Frozen inputs missing. See research/task1_variable_selection_README.md.")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in manifest["files"].items():
        if Path(name).name != name or digest(folder / name) != expected:
            raise ValueError(f"Input integrity check failed: {name}")
    catalog = json.loads((folder / "channel_catalog.json").read_text(encoding="utf-8"))
    if set(catalog) != set(CHANNELS):
        raise ValueError("All eight channels/five families are required.")
    for c, info in catalog.items():
        if (info["family"], info["station"], info["unit"]) != (FAMILY_OF[c], STATIONS[c], UNITS[c]):
            raise ValueError(f"Unexpected channel metadata: {c}")
    frame = pd.read_csv(folder / "observations_long.csv", keep_default_na=False,
                        dtype={"source_file_id": str, "report_id": str})
    return frame, manifest, catalog


def synthetic_input(seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(CALENDAR)
    a, b, noise = rng.normal(size=(3, n))
    for i in range(1, n):
        a[i] += .45 * a[i-1]
        b[i] += .3 * b[i-1]
    phase = 2 * np.pi * np.arange(n) / 365.25
    tmin = 19 + 6*np.sin(phase) + a
    tmax = tmin + 6 + .5*rng.normal(size=n)
    rh_min = np.clip(55 - 8*a + 4*b, 5, 85)
    rh_max = np.clip(rh_min + 16 + rng.normal(size=n), 20, 100)
    rain = np.expm1(np.maximum(0, .5 + .3*b + .2*a))
    values = dict(tmax=tmax, tmin=tmin, rh_min=rh_min, rh_max=rh_max, rainfall=rain,
                  wind_speed=np.maximum(.1, 20+2*noise), cloud=np.clip(60+9*b, 0, 100),
                  sunshine=np.clip(5-.8*b+.3*rng.normal(size=n), 0, 24))
    records = []
    for c, vals in values.items():
        for i, (date, val) in enumerate(zip(CALENDAR, vals)):
            trace = c == "rainfall" and i % 37 == 0
            records.append(dict(date_hkt=date.strftime("%Y-%m-%d"), channel=c, family=FAMILY_OF[c],
                                station=STATIONS[c], unit=UNITS[c], value_text="Trace" if trace else str(val),
                                value_numeric="" if trace else val,
                                qc="RYES_NO_COMPLETENESS_FLAG" if c.startswith("rh_") else "C"))
    catalog = {c: dict(family=FAMILY_OF[c], station=STATIONS[c], unit=UNITS[c]) for c in CHANNELS}
    return pd.DataFrame(records), {"synthetic": True, "seed": seed, "files": {}}, catalog


def prepare_panel(long_frame, trace):
    frame = long_frame.copy()
    frame["date_hkt"] = pd.to_datetime(frame["date_hkt"], format="%Y-%m-%d", errors="raise")
    if frame.duplicated(["date_hkt", "channel"]).any():
        raise ValueError("Duplicate date/channel observations.")
    if set(frame.channel) != set(CHANNELS) or not frame.date_hkt.isin(CALENDAR).all():
        raise ValueError("Unexpected channels or dates; no automatic narrowing.")
    frame["usable"] = False
    frame["analysis_value"] = pd.to_numeric(frame["value_numeric"], errors="coerce")
    is_trace = frame.channel.eq("rainfall") & frame.value_text.str.strip().str.casefold().eq("trace")
    frame.loc[is_trace, "analysis_value"] = trace
    for c in CHANNELS:
        rows = frame.channel.eq(c)
        if not (frame.loc[rows, "unit"].eq(UNITS[c]).all()
                and frame.loc[rows, "station"].eq(STATIONS[c]).all()
                and frame.loc[rows, "family"].eq(FAMILY_OF[c]).all()):
            raise ValueError(f"Row metadata mismatch: {c}")
        v = frame.loc[rows, "analysis_value"]
        qc = frame.loc[rows, "qc"]
        accepted = qc.eq("RYES_NO_COMPLETENESS_FLAG") if c.startswith("rh_") else qc.eq("C")
        valid = np.isfinite(v) & accepted
        if c in ("tmin", "tmax"):
            valid &= v.between(-20, 60)
        elif c in ("rh_min", "rh_max", "cloud"):
            valid &= v.between(0, 100)
        elif c == "sunshine":
            valid &= v.between(0, 24)
        else:
            valid &= v.ge(0)
        frame.loc[rows, "usable"] = valid
    wide = frame.pivot(index="date_hkt", columns="channel", values="analysis_value").reindex(CALENDAR)[CHANNELS]
    usable = frame.pivot(index="date_hkt", columns="channel", values="usable").reindex(CALENDAR).fillna(False).astype(bool)
    wide = wide.where(usable)
    for lo, hi in [("tmin", "tmax"), ("rh_min", "rh_max")]:
        reversed_range = wide[lo].gt(wide[hi])
        wide.loc[reversed_range, [lo, hi]] = np.nan
    if wide.notna().sum().eq(0).any():
        raise ValueError("At least one channel has no valid observations. All families remain required.")
    coverage = []
    for year in range(2022, 2026):
        year_panel = wide.loc[str(year)]
        common = year_panel.notna().all(axis=1)
        for c in CHANNELS:
            source = frame.loc[frame.channel.eq(c) & frame.date_hkt.dt.year.eq(year)]
            coverage.append(dict(year=year, channel=c, family=FAMILY_OF[c], expected_days=len(year_panel),
                source_rows=len(source), usable_days=int(year_panel[c].notna().sum()),
                qc_C=int(source.qc.eq("C").sum()), trace_days=int(
                    source.value_text.str.strip().str.casefold().eq("trace").sum()),
                common_days=int(common.sum()), station=STATIONS[c], unit=UNITS[c]))
    return wide, pd.DataFrame(coverage)


def transform_panel(wide):
    data = wide.copy()
    data["rainfall"] = np.log1p(data["rainfall"])
    return data.dropna(how="any")


def seasonal_basis(dates):
    d = pd.DatetimeIndex(dates)
    phase = 2*np.pi*(d.dayofyear.to_numpy()-1)/(365+d.is_leap_year.astype(int))
    return np.column_stack([np.ones(len(d)), np.sin(phase), np.cos(phase),
                            np.sin(2*phase), np.cos(2*phase)])


def predictor_columns(target):
    return [c for c in CHANNELS if FAMILY_OF[c] != FAMILY_OF[target]]


def rank_guard(matrix, label):
    if not np.isfinite(matrix).all() or np.linalg.matrix_rank(matrix) < matrix.shape[1]:
        raise ValueError(f"Unidentifiable {label}: constant/collinear/nonfinite columns.")
    if np.linalg.cond(matrix) > 1e10:
        raise ValueError(f"Ill-conditioned {label}; no automatic feature deletion.")


def residual_scaling(z, x, y):
    bx, by = np.linalg.lstsq(z, x, rcond=None)[0], np.linalg.lstsq(z, y, rcond=None)[0]
    rx, ry = x-z@bx, y-z@by
    sx, sy = np.sqrt(np.mean(rx**2, axis=0)), float(np.sqrt(np.mean(ry**2)))
    if np.any(sx < 1e-10) or sy < 1e-10:
        raise ValueError("Constant seasonal residual; selection unavailable.")
    return bx, by, sx, sy, rx/sx, ry/sy

## 2. Data contract

- 2022–2025、HKT calendar day；HQ Tmax/Tmin、RH min/max、rainfall、cloud。
- Airport mean wind speed 和 King's Park sunshine 是不同 station 的有限 proxies。
- CSV 只接受 completeness C；RYES 沒有同等 completeness flag，不把它當 C。
- Trace 是 [0,0.05) mm 的 censored observation。0.025 是主 sensitivity 設定；0.05 是上界近似。
- 不 impute 缺失；八個 channels 的共同完整日期固定三方法的 comparison cases。
- 使用來源的 daily date labels 解讀為 HKT 日；各來源的內部 aggregation window 未於本輪另行驗證。
- Rainfall 轉為 log1p(mm)；它不是 PSR probability，更不是 territory-level PSR truth。
- Bounds 或 min/max-order 不符時保留 raw row，但不納入 numerical panel。
- 所有來源、units、QC 和原始文字保留於 inputs；不自動換 station、縮減 families 或加入 pressure/TC。

In [ ]:
LONG, INPUT_MANIFEST, CATALOG = synthetic_input() if SYNTHETIC else load_frozen(INPUT_DIR)
WIDE, COVERAGE = prepare_panel(LONG, TRACE_VALUES["primary"])
print("SYNTHETIC SOFTWARE CHECK ONLY" if SYNTHETIC else "Frozen real observations; no fitting performed yet.")
display(COVERAGE)
print("Common complete dates:", int(WIDE.notna().all(axis=1).sum()), "/", len(CALENDAR))

## 3. Models and validation

各 channel 輪流當 outcome，排除整個 outcome family。Seasonality 控制只在 training fold 擬合。
這避免只靠 Tmin 重建 Tmax，亦避免 shared annual cycle 主導所有 selection。

**F-test：** full OLS 中逐 predictor family joint test。HAC lags 為 7／14 個 retained observations
（不是缺失日期下仍精確七／十四個 calendar days）。每個 scenario × outer fold × HAC lag，
對全部 outcome–predictor-family tests 一起作 BH correction。q<0.05 的 lag-7 groups 用於 F-selected OLS；
partial R² 是 full versus reduced in-sample SSE difference，不是 causal effect。
HAC-robust F-type inference 仍屬近似；不使用 iid nested-model F p-value 充當主推論。

**LASSO：** training residuals 標準化；alpha grid 10^-4 至 1，共 N_ALPHA 個值。
三個 chronological inner folds，按實際日期 purge 七日 gap。以最低平均 validation MSE
加該 alpha 的 fold SE 為界，選最大的 eligible alpha（one-SE rule）。
輸出 channel coefficients；family selected 表示至少一個 channel nonzero，不是 group LASSO。
未在所有 inner folds 收斂的 alpha 不參與 one-SE selection；lasso_cv.csv 保留狀態與原因，
並在下方顯示排除數。沒有有效 alpha 或 final fit 未收斂時，整個 LASSO outcome 標為 unavailable。

**Bayesian selection：** conditional empirical Bayes。Full OLS training residuals 只用 consecutive-day pairs
估 AR(1) rho，限制在 [-0.95,0.95] 並輸出 raw rho、clipping flag 和 whitened residual diagnostic。
按實際日期間隔的 stationary AR(1) covariance whitening；共同 seasonal controls 用 flat prior，
variance 用 1/sigma² prior。對最多四個 predictor families 枚舉全部 subsets。
Zellner g-prior 定義於已 whitening、移除 seasonal controls 的 design。
若 d=n-rank(seasonal controls)、k 為 subset 的 channel 數，則

log BF = -k/2 log(1+g) - d/2 log(1 - g/(1+g) R²)。

Family inclusion priors 獨立；主設定 g=n、pi=0.5。所有 g/n ∈ {0.25,1,4}、
pi ∈ {0.25,0.5} 組合均保留。報 PIP、posterior model weights 和 BMA predictions。
PIP conditional on estimated rho、linear Gaussian/AR(1) assumptions；
沒有積分 rho uncertainty，也不是「該 variable 值得研究的概率」。
與共同 OLS seasonal baseline 的 BMA 差距亦包含 GLS seasonal fitting 差異，
不可將全部 gain 歸因於跨變數資訊。

Outer windows：2022→2023、2022–2023→2024、2022–2024→2025，亦 purge 七日 gap。
每 fold 至少 180 training dates、60 evaluation dates；否則明示 unavailable，不縮窄候選。
2025 已參與先前 EDA。模型比較全部使用同一 outcome、同一日期和 transformed scale。

Sources:
[scikit-learn feature selection](https://scikit-learn.org/stable/modules/feature_selection.html) ·
[LASSO](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) ·
[statsmodels robust covariance](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html) ·
[Bayesian selection under collinearity](https://doi.org/10.1080/00031305.2015.1031827).

In [ ]:
def f_group_tests(z, x, y, columns, lag):
    design = np.column_stack([z, x])
    rank_guard(design, "full OLS design")
    full = sm.OLS(y, design).fit()
    robust = full.get_robustcov_results(cov_type="HAC", maxlags=lag, use_correction=True)
    results = []
    for family in sorted({FAMILY_OF[c] for c in columns}):
        indices = [i for i, c in enumerate(columns) if FAMILY_OF[c] == family]
        restriction = np.zeros((len(indices), design.shape[1]))
        restriction[np.arange(len(indices)), np.array(indices)+z.shape[1]] = 1
        test = robust.f_test(restriction)
        keep = [i for i in range(x.shape[1]) if i not in indices]
        reduced = sm.OLS(y, np.column_stack([z, x[:, keep]])).fit()
        effect = float((reduced.ssr-full.ssr)/reduced.ssr) if reduced.ssr > 1e-12 else np.nan
        pvalue = float(np.asarray(test.pvalue).item())
        if not np.isfinite(pvalue):
            raise ValueError("Nonfinite HAC family test.")
        results.append(dict(predictor_family=family, hac_lag=lag,
                            f_stat=float(np.asarray(test.fvalue).item()), p_value=pvalue,
                            partial_r2=effect, n_train=len(y), design_condition=float(np.linalg.cond(design))))
    return results


def temporal_splits(dates):
    d = pd.DatetimeIndex(dates)
    for train, valid in TimeSeriesSplit(n_splits=3).split(d):
        train = train[d[train] < d[valid[0]]-pd.Timedelta(days=GAP_DAYS)]
        if len(train) < 30 or len(valid) < 20:
            raise ValueError("Insufficient dates for three purged temporal inner folds.")
        assert d[train].max() < d[valid].min()-pd.Timedelta(days=GAP_DAYS)
        yield train, valid


def fit_lasso(z, x, y, znew, xnew, alpha):
    bx, by, sx, sy, rx, ry = residual_scaling(z, x, y)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        fit = Lasso(alpha=float(alpha), fit_intercept=False, max_iter=20000, tol=1e-7).fit(rx, ry)
    prediction = znew@by + sy*((xnew-znew@bx)/sx)@fit.coef_
    return prediction, fit.coef_, fit.coef_*sy/sx


def select_lasso(dates, z, x, y, znew, xnew, n_alpha):
    grid = np.logspace(-4, 0, n_alpha)
    splits = list(temporal_splits(dates))
    scores = np.full((len(grid), len(splits)), np.nan)
    failures = {}
    for j, (train, valid) in enumerate(splits):
        for i, alpha in enumerate(grid):
            try:
                pred, _, _ = fit_lasso(z[train], x[train], y[train], z[valid], x[valid], alpha)
                scores[i, j] = np.mean((y[valid]-pred)**2)
            except (ValueError, np.linalg.LinAlgError, ConvergenceWarning) as exc:
                failures[(i, j)] = str(exc)
    complete = np.isfinite(scores).all(axis=1)
    if not complete.any():
        raise ValueError("No alpha converged in all three inner folds.")
    means, ses = np.full(len(grid), np.nan), np.full(len(grid), np.nan)
    means[complete] = scores[complete].mean(axis=1)
    ses[complete] = scores[complete].std(axis=1, ddof=1)/np.sqrt(len(splits))
    eligible = np.flatnonzero(complete)
    best = int(eligible[np.argmin(means[complete])])
    chosen = int(np.flatnonzero(complete & (means <= means[best]+ses[best]))[-1])
    prediction, coef, native_coef = fit_lasso(z, x, y, znew, xnew, grid[chosen])
    records = [dict(alpha=float(a), mean_validation_mse=float(means[i]), fold_se=float(ses[i]),
                    selected=bool(i == chosen), status="ok" if complete[i] else "unavailable",
                    failure_reason=" | ".join(f"fold {j+1}: {failures[(i,j)]}"
                                              for j in range(len(splits)) if (i,j) in failures),
                    **{f"fold_{j+1}_mse": float(s) for j, s in enumerate(scores[i])})
               for i, a in enumerate(grid)]
    return prediction, coef, native_coef, float(grid[chosen]), records


def ar1_whiten(a, dates, rho):
    values = np.asarray(a, dtype=float)
    original_1d = values.ndim == 1
    values = values[:, None] if original_1d else values
    gaps = np.diff(pd.DatetimeIndex(dates).values).astype("timedelta64[D]").astype(int)
    if np.any(gaps < 1) or not (-1 < rho < 1):
        raise ValueError("AR(1) requires increasing distinct dates and stationary rho.")
    out = np.empty_like(values)
    out[0] = np.sqrt(1-rho*rho)*values[0]
    powers = rho**gaps
    scale = np.sqrt((1-rho**(2*gaps))/(1-rho*rho))
    out[1:] = (values[1:]-powers[:, None]*values[:-1])/scale[:, None]
    return out[:, 0] if original_1d else out


def bayesian_setup(dates, z, x, y):
    design = np.column_stack([z, x])
    rank_guard(design, "Bayesian full design")
    residual = y-design@np.linalg.lstsq(design, y, rcond=None)[0]
    consecutive = np.diff(pd.DatetimeIndex(dates).values).astype("timedelta64[D]").astype(int) == 1
    prev, current = residual[:-1][consecutive], residual[1:][consecutive]
    if len(prev) < 30 or prev@prev < 1e-12:
        raise ValueError("Cannot estimate AR(1) from training residuals.")
    raw_rho = float(prev@current/(prev@prev))
    rho = float(np.clip(raw_rho, -.95, .95))
    wz, wx, wy = (ar1_whiten(a, dates, rho) for a in (z, x, y))
    rank_guard(np.column_stack([wz, wx]), "whitened full design")
    bx, by = np.linalg.lstsq(wz, wx, rcond=None)[0], np.linalg.lstsq(wz, wy, rcond=None)[0]
    rx, ry = wx-wz@bx, wy-wz@by
    sst = float(ry@ry)
    if sst < 1e-12:
        raise ValueError("No residual outcome variation for Bayesian selection.")
    white_residual = ar1_whiten(residual, dates, rho)
    acf = float(np.corrcoef(white_residual[:-1][consecutive], white_residual[1:][consecutive])[0, 1])
    return dict(wz=wz, wx=wx, wy=wy, rx=rx, ry=ry, sst=sst,
                d=len(y)-np.linalg.matrix_rank(wz), rho=rho, raw_rho=raw_rho,
                rho_clipped=bool(rho != raw_rho), whitened_residual_lag1=acf)


def bayesian_select(setup, columns, znew, xnew, g, inclusion_prior):
    families = sorted({FAMILY_OF[c] for c in columns})
    logs, predictions, masks, records = [], [], [], []
    for mask in itertools.product([False, True], repeat=len(families)):
        chosen = [f for f, include in zip(families, mask) if include]
        indices = [i for i, c in enumerate(columns) if FAMILY_OF[c] in chosen]
        beta = np.zeros(len(columns))
        if indices:
            fit = np.linalg.lstsq(setup["rx"][:, indices], setup["ry"], rcond=None)[0]
            explained = setup["rx"][:, indices]@fit
            r2 = float(explained@explained/setup["sst"])
            if r2 < -1e-8 or r2 > 1+1e-8:
                raise ValueError("Invalid Bayesian residual R².")
            r2 = float(np.clip(r2, 0, 1))
            beta[indices] = g/(1+g)*fit
        else:
            r2 = 0.0
        log_bf = -.5*len(indices)*np.log1p(g)-.5*setup["d"]*np.log1p(-g/(1+g)*r2)
        log_prior = len(chosen)*np.log(inclusion_prior)+(len(families)-len(chosen))*np.log1p(-inclusion_prior)
        gamma = np.linalg.lstsq(setup["wz"], setup["wy"]-setup["wx"]@beta, rcond=None)[0]
        predictions.append(znew@gamma+xnew@beta)
        logs.append(log_bf+log_prior)
        masks.append(mask)
        records.append(dict(subset="|".join(chosen) or "null", n_channels=len(indices),
                            log_bayes_factor=float(log_bf), r2=r2))
    weights = np.exp(np.array(logs)-logsumexp(logs))
    assert np.isclose(weights.sum(), 1) and np.all(weights >= 0)
    pips = np.array(masks, float).T@weights
    for record, weight in zip(records, weights):
        record["posterior_weight"] = float(weight)
    return weights@np.array(predictions), dict(zip(families, pips)), records

In [ ]:
def score_with_blocks(dates, truth, prediction, baseline, block_length, draws, seed):
    dates = pd.DatetimeIndex(dates)
    loss, base_loss = (truth-prediction)**2, (truth-baseline)**2
    mse, base_mse = float(np.mean(loss)), float(np.mean(base_loss))
    calendar = pd.date_range(f"{dates[0].year}-01-01", f"{dates[0].year}-12-31")
    locations = calendar.get_indexer(dates)
    if np.any(locations < 0) or len(set(locations)) != len(locations):
        raise ValueError("Bootstrap requires unique dates in one evaluation year.")
    daily = np.full((len(calendar), 2), np.nan)
    daily[locations] = np.column_stack([loss, base_loss])
    rng = np.random.default_rng(seed)
    starts = rng.integers(0, len(calendar), size=(draws, math.ceil(len(calendar)/block_length)))
    positions = (starts[..., None]+np.arange(block_length)) % len(calendar)
    positions = positions.reshape(draws, -1)[:, :len(calendar)]
    selected = daily[positions]
    counts = np.sum(np.isfinite(selected[:, :, 0]), axis=1)
    sums = np.nansum(selected, axis=1)
    mean_losses = np.divide(sums, counts[:, None], out=np.full_like(sums, np.nan), where=counts[:, None] > 0)
    deltas = mean_losses[:, 1]-mean_losses[:, 0]
    skills = np.divide(deltas, mean_losses[:, 1], out=np.full(draws, np.nan), where=mean_losses[:, 1] > 1e-12)
    finite = np.isfinite(skills)
    enough = int(finite.sum()) >= math.ceil(.9*draws)
    skill_ci = np.quantile(skills[finite], [.025, .975]) if enough else [np.nan, np.nan]
    delta_ok = np.isfinite(deltas)
    delta_ci = np.quantile(deltas[delta_ok], [.025, .975]) if delta_ok.sum() >= .9*draws else [np.nan, np.nan]
    return dict(n_test=len(truth), mse=mse, baseline_mse=base_mse, delta_mse=base_mse-mse,
                skill=1-mse/base_mse if base_mse > 1e-12 else np.nan,
                skill_ci_low=float(skill_ci[0]), skill_ci_high=float(skill_ci[1]),
                delta_mse_ci_low=float(delta_ci[0]), delta_mse_ci_high=float(delta_ci[1]),
                block_calendar_days=block_length, valid_bootstrap_draws=int(finite.sum()),
                bootstrap_status="ok" if enough else "unavailable: zero baseline loss / too few finite resamples")


def run_experiments(wide_scenarios, n_alpha=N_ALPHA, draws=N_BOOTSTRAP):
    rows = {k: [] for k in ("f_tests", "lasso_coefficients", "lasso_cv", "bayesian_pips",
                            "bayesian_models", "predictions", "metrics", "status")}
    for scenario, wide in wide_scenarios.items():
        data = transform_panel(wide)
        for year in (2023, 2024, 2025):
            print(f"{scenario}: evaluation year {year}")
            test = data[data.index.year == year]
            training = data[data.index < pd.Timestamp(f"{year}-01-01")-pd.Timedelta(days=GAP_DAYS)]
            contexts, fold_tests = {}, []
            for target in CHANNELS:
                key = dict(scenario=scenario, evaluation_year=year, target=target)
                try:
                    if len(training) < MIN_TRAIN or len(test) < MIN_TEST:
                        raise ValueError(f"Insufficient common dates: train={len(training)}, test={len(test)}")
                    columns = predictor_columns(target)
                    assert all(FAMILY_OF[c] != FAMILY_OF[target] for c in columns)
                    z, znew = seasonal_basis(training.index), seasonal_basis(test.index)
                    x, xnew = training[columns].to_numpy(), test[columns].to_numpy()
                    y, truth = training[target].to_numpy(), test[target].to_numpy()
                    rank_guard(np.column_stack([z, x]), "full design")
                    residual_scaling(z, x, y)
                    local_tests = []
                    for lag in (7, 14):
                        local_tests.extend([dict(**key, **r) for r in f_group_tests(z, x, y, columns, lag)])
                    fold_tests.extend(local_tests)
                    contexts[target] = (columns, z, znew, x, xnew, y, truth)
                except (ValueError, np.linalg.LinAlgError) as exc:
                    rows["status"].append(dict(**key, method="all", status="unavailable", reason=str(exc)))
            for lag in (7, 14):
                selected = [r for r in fold_tests if r["hac_lag"] == lag]
                if selected:
                    q_values = multipletests([r["p_value"] for r in selected], method="fdr_bh")[1]
                    for record, q in zip(selected, q_values):
                        record["q_value"] = float(q)
                        record["bh_test_count"] = len(selected)
            rows["f_tests"].extend(fold_tests)
            for target, (columns, z, znew, x, xnew, y, truth) in contexts.items():
                key = dict(scenario=scenario, evaluation_year=year, target=target)
                baseline = znew@np.linalg.lstsq(z, y, rcond=None)[0]
                predictions = {"seasonal_baseline": baseline}
                chosen_families = {r["predictor_family"] for r in fold_tests
                                   if r["target"] == target and r["hac_lag"] == 7 and r["q_value"] < .05}
                selected = [i for i, c in enumerate(columns) if FAMILY_OF[c] in chosen_families]
                fit_x = np.column_stack([z, x[:, selected]])
                predictions["f_selected_ols"] = np.column_stack([znew, xnew[:, selected]])@np.linalg.lstsq(fit_x, y, rcond=None)[0]
                try:
                    pred, coef, native_coef, alpha, cv = select_lasso(training.index, z, x, y, znew, xnew, n_alpha)
                    predictions["lasso"] = pred
                    for c, beta, native in zip(columns, coef, native_coef):
                        rows["lasso_coefficients"].append(dict(**key, predictor=c, predictor_family=FAMILY_OF[c],
                            coefficient_standardized=float(beta), coefficient_transformed_units=float(native),
                            selected=bool(abs(beta) > 1e-10), alpha=alpha))
                    rows["lasso_cv"].extend([dict(**key, **r) for r in cv])
                except (ValueError, np.linalg.LinAlgError, ConvergenceWarning) as exc:
                    rows["status"].append(dict(**key, method="lasso", status="unavailable", reason=str(exc)))
                try:
                    setup = bayesian_setup(training.index, z, x, y)
                    for g_factor, pi in itertools.product((.25, 1., 4.), (.25, .5)):
                        pred, pips, models = bayesian_select(setup, columns, znew, xnew, len(y)*g_factor, pi)
                        bayes_key = dict(**key, g_factor=g_factor, inclusion_prior=pi,
                                         rho=setup["rho"], raw_rho=setup["raw_rho"],
                                         rho_clipped=setup["rho_clipped"],
                                         whitened_residual_lag1=setup["whitened_residual_lag1"])
                        rows["bayesian_pips"].extend([dict(**bayes_key, predictor_family=f, pip=float(p)) for f, p in pips.items()])
                        rows["bayesian_models"].extend([dict(**bayes_key, **r) for r in models])
                        if g_factor == 1 and pi == .5:
                            predictions["bayesian_bma"] = pred
                except (ValueError, np.linalg.LinAlgError) as exc:
                    rows["status"].append(dict(**key, method="bayesian_bma", status="unavailable", reason=str(exc)))
                for method, prediction in predictions.items():
                    if not np.isfinite(prediction).all():
                        rows["status"].append(dict(**key, method=method, status="unavailable", reason="Nonfinite predictions"))
                        continue
                    rows["status"].append(dict(**key, method=method, status="ok", reason="",
                                               n_train=len(y), n_test=len(truth),
                                               last_train_date=training.index.max().date().isoformat(),
                                               first_test_date=test.index.min().date().isoformat()))
                    for date, value, pred, base in zip(test.index, truth, prediction, baseline):
                        rows["predictions"].append(dict(**key, method=method, date_hkt=date.date().isoformat(),
                            observed_transformed=float(value), predicted_transformed=float(pred),
                            seasonal_baseline_transformed=float(base)))
                    for block in (7, 14):
                        rows["metrics"].append(dict(**key, method=method,
                            outcome_scale="log1p(mm)" if target == "rainfall" else UNITS[target],
                            **score_with_blocks(test.index, truth, prediction, baseline, block, draws,
                                               SEED+year*17+CHANNELS.index(target))))
    return {name: pd.DataFrame(records) for name, records in rows.items()}

## 4. Run all experiments

下列 cell 才開始 fitting。三個 Trace scenarios、三個 outer windows、八個 outcomes 均執行。
不使用 random split；three-method selection 結果不互換成共同 importance score。

MSE skill = 1 − model MSE / seasonal-baseline MSE。正值表示 reconstruction 勝過 seasonal-only。
配對 CI 使用同一 outcome、同一 evaluation 日期，七／十四日 circular calendar blocks，
保留缺失日期位置；區間只反映 evaluation cases 的 sampling uncertainty，沒有重新訓練模型。
Rainfall MSE 是 log1p scale；不同 units 的 MSE 不可直接排名。

In [ ]:
PANELS = {name: prepare_panel(LONG, value)[0] for name, value in TRACE_VALUES.items()}
with threadpool_limits(limits=1):
    RESULTS = run_experiments(PANELS)
display(RESULTS["status"].groupby(["scenario", "method", "status"], dropna=False).size().rename("count").reset_index())
if not RESULTS["lasso_cv"].empty:
    print("Excluded LASSO alpha candidates (any inner fold failed):",
          int(RESULTS["lasso_cv"].status.ne("ok").sum()))
display(RESULTS["metrics"].query("scenario == 'primary' and block_calendar_days == 7").head(16))

In [ ]:
def save_figures(results, wide, output):
    prefix = "SYNTHETIC CHECK — " if SYNTHETIC else ""
    figures = []
    descriptive = transform_panel(wide)
    descriptive = descriptive[descriptive.index.year <= 2024]
    z = seasonal_basis(descriptive.index)
    vals = descriptive.to_numpy()
    residual = vals-z@np.linalg.lstsq(z, vals, rcond=None)[0]
    correlations = {
        "raw_2022_2024": descriptive.corr(),
        "season_adjusted_2022_2024": pd.DataFrame(residual, columns=CHANNELS).corr(),
    }
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")
    for ax, (name, corr) in zip(axes, correlations.items()):
        im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set(xticks=range(8), yticks=range(8), xticklabels=CHANNELS, yticklabels=CHANNELS,
               title=prefix+name.replace("_", " "))
        ax.tick_params(axis="x", labelrotation=55)
        for i in range(8):
            for j in range(8):
                ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center",
                        color="white" if abs(corr.iloc[i,j]) > .65 else "black", fontsize=8)
        fig.colorbar(im, ax=ax, label="Pearson correlation")
        corr.to_csv(output / (name+".csv"))
    fig.suptitle("Common complete dates; rainfall = log1p(mm); observations from three stations", fontsize=11)
    fig.savefig(output / "correlation_heatmaps.png")
    figures.append(fig)

    ftests, lasso, bayes = (results[k] for k in ("f_tests", "lasso_coefficients", "bayesian_pips"))
    if not (ftests.empty or lasso.empty or bayes.empty):
        f = ftests.query("scenario == 'primary' and evaluation_year == 2025 and hac_lag == 7").copy()
        f["value"] = (f.q_value < .05).astype(float)
        l = lasso.query("scenario == 'primary' and evaluation_year == 2025").groupby(
            ["target", "predictor_family"], as_index=False)["selected"].max().rename(columns={"selected": "value"})
        b = bayes.query("scenario == 'primary' and evaluation_year == 2025 and g_factor == 1 and inclusion_prior == .5").rename(columns={"pip": "value"})
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), layout="constrained")
        for ax, frame, label in zip(axes, [f, l, b], ["F: q<0.05", "LASSO: any channel nonzero", "Bayesian: PIP"]):
            matrix = frame.pivot(index="predictor_family", columns="target", values="value").reindex(
                index=list(FAMILIES), columns=CHANNELS).astype(float)
            cmap = plt.colormaps["Blues"].copy()
            cmap.set_bad("#dddddd")
            im = ax.imshow(matrix.to_numpy(), vmin=0, vmax=1, cmap=cmap, aspect="auto")
            ax.set(xticks=range(8), yticks=range(5), xticklabels=CHANNELS, yticklabels=list(FAMILIES),
                   title=prefix+label, xlabel="Outcome (own family excluded)")
            ax.tick_params(axis="x", labelrotation=55)
            for i in range(5):
                for j in range(8):
                    v = matrix.iloc[i, j]
                    ax.text(j, i, "—" if pd.isna(v) else f"{v:.2f}", ha="center", va="center",
                            fontsize=8, color="white" if pd.notna(v) and v > .65 else "black")
            fig.colorbar(im, ax=ax, label=label)
        fig.suptitle("2022–2024 training → 2025 evaluation; methods measure different quantities", fontsize=11)
        fig.savefig(output / "selection_matrix_2025.png")
        figures.append(fig)

    metrics = results["metrics"]
    if not metrics.empty:
        m = metrics.query("scenario == 'primary' and evaluation_year == 2025 and block_calendar_days == 7 and method != 'seasonal_baseline'")
        fig, axes = plt.subplots(2, 4, figsize=(16, 8), layout="constrained")
        for ax, target in zip(axes.flat, CHANNELS):
            part = m[m.target.eq(target)]
            for i, method in enumerate(COLORS):
                row = part[part.method.eq(method)]
                if not row.empty:
                    r = row.iloc[0]
                    ax.vlines(i, r.skill_ci_low, r.skill_ci_high, color=COLORS[method], linewidth=2)
                    ax.plot(i, r.skill, "o", color=COLORS[method])
            ax.axhline(0, color="#666666", linewidth=.8, linestyle="--")
            ax.set(title=target + (" [log1p rainfall]" if target == "rainfall" else ""),
                   xticks=range(3), xticklabels=["F-selected", "LASSO", "BMA"], ylabel="MSE skill vs seasonal-only")
        fig.suptitle(prefix+"2025 reconstruction; 95% paired 7-calendar-day block CI", fontsize=13)
        fig.savefig(output / "reconstruction_skill_2025.png")
        figures.append(fig)
    return figures


def candidate_questions(results):
    prompts = {
        "temperature": "Which temperature regimes and spatial contrasts deserve further forecast evaluation?",
        "humidity": "Does RH contribute moisture-state information beyond temperature and rainfall?",
        "rainfall": "Which episodic rainfall regimes differ from other daily weather channels?",
        "wind": "Would wind speed/direction evaluation add a useful dimension after measurement alignment?",
        "weather": "Which sky/phenomenon labels merit an explicit multi-event truth ontology?",
    }
    limits = {
        "temperature": "HQ extrema are a station reference, not automatic territory truth.",
        "humidity": "HQ RYES endpoints lack CSV completeness flags; this is not the regional sampled envelope.",
        "rainfall": "HQ gauge is not territory PSR truth; log-scale reconstruction downweights extremes.",
        "wind": "Airport daily mean speed excludes direction, gusts and forecast text qualifiers.",
        "weather": "Cloud/sunshine at HQ/King's Park cover only part of weather descriptions.",
    }
    rows = []
    for family in FAMILIES:
        f = results["f_tests"]
        l = results["lasso_coefficients"]
        b = results["bayesian_pips"]
        f = f[f.predictor_family.eq(family) & f.scenario.eq("primary") & f.hac_lag.eq(7)] if not f.empty else f
        l = l[l.predictor_family.eq(family) & l.scenario.eq("primary")] if not l.empty else l
        b = b[b.predictor_family.eq(family) & b.scenario.eq("primary") & b.g_factor.eq(1) & b.inclusion_prior.eq(.5)] if not b.empty else b
        rows.append(dict(family=family,
            f_outcomes_in_any_fold="|".join(sorted(f.loc[f.q_value < .05, "target"].unique())) if not f.empty else "",
            lasso_outcomes_in_any_fold="|".join(sorted(l.loc[l.selected, "target"].unique())) if not l.empty else "",
            bayesian_pip_ge_half_outcomes_in_any_fold="|".join(sorted(b.loc[b.pip >= .5, "target"].unique())) if not b.empty else "",
            follow_up_question=prompts[family], measurement_limit=limits[family],
            interpretation="Candidate for review, not an importance rank or three-method vote"))
    return pd.DataFrame(rows)

## 5. Save results and figures

Selection panels 的數字語義不同，不能相加或投票。灰色表示 own-family 排除或 unavailable；
若某模型 unavailable，請同時讀 status.csv，不能把缺失當成未選中。

回傳 ZIP 包含 numerical outputs、predictions、settings、versions、input hashes 和 plots；
不包含 credentials 或 PostgreSQL connection/session files。缺少 results 時會保留 unavailable 原因。

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = DATA_DIR / ("synthetic_validation" if SYNTHETIC else "results") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
for name, table in RESULTS.items():
    table.to_csv(OUTPUT_DIR / (name+".csv"), index=False)
COVERAGE.to_csv(OUTPUT_DIR / "coverage.csv", index=False)
CANDIDATES = candidate_questions(RESULTS)
CANDIDATES.to_csv(OUTPUT_DIR / "research_candidates.csv", index=False)
json_save(OUTPUT_DIR / "input_manifest.json", INPUT_MANIFEST)
json_save(OUTPUT_DIR / "channel_catalog.json", CATALOG)
json_save(OUTPUT_DIR / "settings.json", dict(synthetic=SYNTHETIC, seed=SEED, bootstrap_draws=N_BOOTSTRAP,
    alpha_grid_size=N_ALPHA, trace_values_mm=TRACE_VALUES, min_train=MIN_TRAIN, min_test=MIN_TEST,
    gap_calendar_days=GAP_DAYS, outer_evaluation_years=[2023, 2024, 2025],
    interpretation="Ex-post reconstruction; no causal/social/forecast-accuracy ranking; 2025 previously explored"))
packages = ["numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "matplotlib", "nbformat", "nbclient", "ipykernel", "jupyterlab"]
json_save(OUTPUT_DIR / "versions.json", {p: importlib.metadata.version(p) for p in packages})
FIGURES = save_figures(RESULTS, WIDE, OUTPUT_DIR)
for figure in FIGURES:
    display(figure)
    plt.close(figure)
if not RESULTS["status"].empty:
    unavailable = RESULTS["status"].query("status == 'unavailable'")
    print("Unavailable method/outcome/folds:", len(unavailable))
    if len(unavailable):
        display(unavailable)
display(CANDIDATES)
json_save(OUTPUT_DIR / "result_manifest.json", {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "synthetic": SYNTHETIC,
    "files": {p.name: digest(p) for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()}})
ZIP_PATH = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
print("Return this results ZIP:", ZIP_PATH)

## 6. Interpretation checklist

- 先讀 coverage/status，再對照三種 selection evidence；不選一個「best method」後忽略其他結果。
- 檢查跨年份、Trace bounds、HAC lag 和 Bayesian prior sensitivity；不要只看 2025 一張圖。
- High reconstruction skill 是特定線性任務下的可重建性，不是 HKO forecast skill。
- Low skill 或未選中可能來自獨特資訊、noise、非線性、measurement mismatch 或低 power。
- 同時有 predictive contribution 與 remaining unexplained variation，仍不足以證明研究或社會重要性。
- Family channel 數不同；本設計沒有證明 selection frequency 可作跨 family 公平排名。
- research_candidates.csv 是後續問題清單，沒有修改既有選題。應連同所有 outcomes、methods、folds 解讀。
- Statistical significance、nonzero coefficient、PIP 分別回應不同問題；不合成總分、不作因果主張。
- Linear/Gaussian/AR(1) 對 rainfall、bounded RH/cloud 可能不充分；優先檢查 held-period skill 和分歧。

完成後回傳上列 ZIP；保留 notebook executed outputs 方便你自行檢視。